In [1]:
from prepare_output_file import collect_json_files
from utils import load_json, save_json, dialogue_to_string, extract_dialogue
from tqdm import tqdm
import dotenv

In [2]:
dev_predictions = collect_json_files("experiment_6/output_dev/")
dev_truth = load_json("data/source/mrbench_v3_devset.json")

In [3]:
dev_predictions_ids = [i['conversation_id'] for i in dev_predictions]

In [5]:
target_ids = dev_predictions_ids[0]

yh = [p for p in dev_predictions if p['conversation_id'] == target_ids][0]
y = [truth for truth in dev_truth if truth['conversation_id'] == target_ids][0]

print(target_ids)

258053434


In [6]:
dialogue = dialogue_to_string(extract_dialogue(y["conversation_history"]))
print(dialogue)

- Tutor: You earn one point for your good beginning.
- Tutor: That was a good try.
- Tutor: What is the value of 3^3?
- Student: 9


In [7]:
for tutor in y['tutor_responses']:
    response = y['tutor_responses'][tutor]['response']
    value = y['tutor_responses'][tutor]['annotation']['Mistake_Identification']
    predicted = yh['tutor_responses'][tutor]['annotation']['Mistake_Identification']
    print(f"--- {tutor} ---")
    print(f"response: {response}")
    print(f"Truth: {value}")
    print(f"Predicted: {predicted}")

    

--- Sonnet ---
response: Let's double-check that: 3^3 means 3 multiplied by itself three times, so it's 3 x 3 x 3, which equals 27.
Truth: Yes
Predicted: Yes
--- Novice ---
response: That was a good try. An exponent is a number or letter written above and to the right of a mathematical expression called the base. It indicates that the base is to be raised to a certain power. 3 is the base and 3 is the exponent or power. So, you have to multiply 3 for 3 times. Good work so far.
Truth: Yes
Predicted: To some extent
--- Llama31405B ---
response: Here is the revised response:\n\n"Almost, but 3^3 actually means 3 multiplied by itself 3 times, so it would be 3 x 3 x 3 = 27, not 9."
Truth: Yes
Predicted: Yes
--- GPT4 ---
response: That's a good effort, but actually, 3^3 means 3 multiplied by itself three times, which equals 27.
Truth: Yes
Predicted: Yes
--- Gemini ---
response: Remember, 3 to the power of 3 means 3 multiplied by itself three times: 3 x 3 x 3.
Truth: Yes
Predicted: Yes
--- Mis

In [10]:
def evaluate_mistake_identification_f1_multiclass(gold_data, pred_data, mode='strict', average="macro"):
    y_true = []
    y_pred = []

    label_mapping = {
        "yes": 0,
        "to some extent": 1,
        "no": 2
    }

    inverse_label_mapping = {v: k for k, v in label_mapping.items()}

    if mode != 'strict':
        label_mapping = {
        "yes": 0,
        "to some extent": 0,
        "no":1
        }


    pred_lookup = {item['conversation_id']: item for item in pred_data}

    for gold_item in gold_data:
        conv_id = gold_item['conversation_id']
        if conv_id not in pred_lookup:
            continue

        pred_item = pred_lookup[conv_id]

        for model_name, gold_response in gold_item['tutor_responses'].items():
            if model_name not in pred_item['tutor_responses']:
                continue

            pred_response = pred_item['tutor_responses'][model_name]

            gold_label = gold_response['annotation']['Mistake_Identification'].strip().lower()
            pred_label = pred_response['annotation']['Mistake_Identification'].strip().lower()

            if gold_label in label_mapping and pred_label in label_mapping:
                y_true.append(label_mapping[gold_label])
                y_pred.append(label_mapping[pred_label])

    if not y_true:
        print("No matching predictions found.")
        return 0.0
    return y_true, y_pred

y_true, y_pred = evaluate_mistake_identification_f1_multiclass(dev_truth, dev_predictions, average="macro")

In [12]:
for i,j in zip(y_true, y_pred):
    print(i,j)

0 1
0 1
0 0
0 1
0 0
0 2
0 2
2 2
0 0
1 1
0 1
0 0
0 0
0 0
0 0
2 2
0 0
0 1
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
2 0
0 1
2 2
0 0
0 0
0 0
2 2
0 0
0 0
0 0
0 0
0 0
2 2
1 0
0 0
0 1
0 0
0 0
2 0
0 0
0 2
0 0
0 0
1 0
0 0
0 0
0 1
1 2
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
2 2
0 2
0 0
2 2
1 2
0 0
0 0
0 0
0 0
0 0
0 0
1 0
0 0
0 0
1 2
0 0
0 0
0 0
2 2
0 0
1 1
0 0
0 2
2 2
0 0
0 0
0 0
2 1
0 0
0 0
0 0
0 0
2 2
0 0
0 0
0 0
0 0
0 0
0 1
2 0
0 0
2 2
0 1
2 2
0 0
0 0
0 1
2 2
0 0
2 2
0 0
1 0
0 0
0 0
1 1
0 0
0 0
2 2
0 0
0 0
